# TIGER: Generative Retrieval with Semantic IDs

This tutorial walks through the TIGER generative retrieval model for sequential recommendation
using the [MovieLens 20M](https://grouplens.org/datasets/movielens/20m/) dataset.

TIGER represents items as **Semantic IDs** -- tuples of discrete codes produced by RQ-VAE.
A Transformer encoder-decoder is then trained to predict the next item's Semantic ID given a user's interaction history.

### Pipeline overview

1. Download and prepare ML-20M interaction data
2. Apply k-core filtering
3. Train a `SIDTokenizer` (with random embeddings for this demo)
4. Create a `TIGERModel` with the tokenizer
5. Split data with `loo_split` (leave-one-out)
6. Train with `fit(train_df, val_df)`
7. Evaluate with `evaluate(test_df)`
8. Generate recommendations with `predict(interactions)`
9. Save / load the trained model

## Setup

In [1]:
import numpy as np
import pandas as pd
import warnings

from rectools.semantic.data_handling import k_core, loo_split
from rectools.semantic.tokenizer import SIDTokenizer
from rectools.semantic.tiger import TIGERModel

warnings.simplefilter("ignore")

## 1. Download and prepare ML-20M

We download the MovieLens 20M dataset and prepare it as a DataFrame with
`user_id`, `item_id`, and `timestamp` columns.

In [2]:
%%time
!wget -q https://files.grouplens.org/datasets/movielens/ml-20m.zip -O ml-20m.zip
!unzip -o -q ml-20m.zip
!rm ml-20m.zip

[ml-20m.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of ml-20m.zip or
        ml-20m.zip.zip, and cannot find ml-20m.zip.ZIP, period.
CPU times: user 8.84 ms, sys: 41.8 ms, total: 50.7 ms
Wall time: 794 ms


In [3]:
ratings = pd.read_csv("ml-20m/ratings.csv")
ratings.columns = ["user_id", "item_id", "rating", "timestamp"]

# Keep only positive interactions (rating >= 4)
interactions = ratings[ratings["rating"] >= 4][["user_id", "item_id", "timestamp"]].copy()
interactions = interactions.sort_values(["user_id", "timestamp"]).reset_index(drop=True)

print(f"Interactions: {len(interactions):,}")
print(f"Users: {interactions['user_id'].nunique():,}, Items: {interactions['item_id'].nunique():,}")
interactions.head()

Interactions: 9,995,410
Users: 138,287, Items: 20,720


,user_id,item_id,timestamp
0,1,1079,1094785665
1,1,2959,1094785698
2,1,3996,1094785727
3,1,151,1094785734
4,1,1374,1094785746


In [4]:
movies = pd.read_csv("ml-20m/movies.csv")
movies.columns = ["item_id", "title", "genres"]
movies.head()

,item_id,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


## 2. K-core filtering

`k_core()` iteratively removes users and items with fewer than the specified minimum interactions.
This ensures that every user and item has enough signal for sequential modeling.

In [5]:
interactions = k_core(
    interactions,
    user_min_interactions=5,
    item_min_interactions=5,
)

print(f"After k-core: {len(interactions):,} interactions")
print(f"Users: {interactions['user_id'].nunique():,}, Items: {interactions['item_id'].nunique():,}")

After k-core: 9,977,451 interactions
Users: 136,674, Items: 13,680


## 3. Train a tokenizer

The `SIDTokenizer` maps each item ID to a **Semantic ID** -- a tuple of discrete codes.
It uses RQ-VAE (or RK-means) to quantize item embeddings into hierarchical codebooks.

### Embedding items with sentence-transformers

Here we embed item metadata (title, genres, description) using
a sentence-transformers model and pass the resulting embeddings to `SIDTokenizer.fit()`:

In [6]:
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import trange

def encode_texts(texts: list, model_name: str, device: str = "cpu", batch_size: int = 256) -> np.ndarray:
    """Encode texts into embeddings using a SentenceTransformer model.

    Parameters
    ----------
    texts : list of str
        Texts to encode.
    model_name : str
        Name or path of a sentence-transformers model.
    device : str
        Device to run the model on.
    batch_size : int
        Batch size for encoding.

    Returns
    -------
    np.ndarray
        Embedding matrix of shape ``(len(texts), embed_dim)``.
    """
    embeds = []
    model = SentenceTransformer(model_name).to(device).eval()
    with torch.no_grad():
        for start in trange(0, len(texts), batch_size, desc="Embedding the metadata"):
            end = min(start + batch_size, len(texts))
            embeds.append(model.encode(texts[start:end]))

    return np.vstack(embeds)

In [7]:
from rectools.semantic.tokenizer import SIDTokenizer

# Build text descriptions from movie metadata
meta = movies[movies["item_id"].isin(interactions["item_id"])].copy()
meta["text"] = meta["title"] + " " + meta["genres"].str.replace("|", " ")

# Encode with a sentence-transformers model
embeddings = encode_texts(
    meta["text"].tolist(),
    model_name="Qwen/Qwen3-Embedding-0.6B",
    device="cuda",
)

# Train the tokenizer
tokenizer = SIDTokenizer(
    input_dim=embeddings.shape[1],
    codebook_sizes=[256, 256, 256],
    codebook_dim=32,
    quantizer="rqvae",
    device="cuda",
)
tokenizer.fit(
    item_ids=meta["item_id"].tolist(),
    embeddings=embeddings,
    max_epochs=100,
    patience=10,
)

print(f"Tokenizer vocabulary: {len(tokenizer)} unique SIDs")

Embedding the metadata:   0%|          | 0/54 [00:00<?, ?it/s]

Initializing codebooks with constrained k-means:   0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/700 [00:00<?, ?it/s]

Early stopping at epoch 22 (patience exceeded 10)
Best model's collision rate: 0.42%
Tokenizer vocabulary: 13651 unique SIDs


Let's see which items get duplicate semantic IDs:

In [ ]:
from utils import find_conflicts_df

find_conflicts_df(tokenizer.id2sid, meta)

,SID,item_id,text
0,"(235, 229, 202)",421,Black Beauty (1994) Adventure Children Drama
1,"(235, 229, 202)",986,Fly Away Home (1996) Adventure Children
2,"(10, 64, 101)",502,"Next Karate Kid, The (1994) Action Children Ro..."
3,"(10, 64, 101)",2422,"Karate Kid, Part III, The (1989) Action Advent..."
4,"(15, 132, 12)",913,"Maltese Falcon, The (1941) Film-Noir Mystery"
5,"(15, 132, 12)",8228,"Maltese Falcon, The (a.k.a. Dangerous Female) ..."
6,"(208, 107, 255)",1007,"Apple Dumpling Gang, The (1975) Children Comed..."
7,"(208, 107, 255)",2016,"Apple Dumpling Gang Rides Again, The (1979) Ch..."
8,"(99, 84, 111)",1644,I Know What You Did Last Summer (1997) Horror ...
9,"(99, 84, 111)",2338,I Still Know What You Did Last Summer (1998) H...


## 4. Create a TIGERModel

The `TIGERModel` wraps the TIGER Transformer encoder-decoder.
It accepts a pretrained tokenizer and model/training hyperparameters.

In [9]:
tiger = TIGERModel(
    tokenizer=tokenizer,
    hidden_units=128,
    num_blocks=4,
    num_heads=6,
    dropout_rate=0.1,
    max_length=20,
    # Training hyperparams
    lr=1e-3,
    lr_schedule="cosine",
    max_epochs=3,
    patience=None,
    batch_size=256,
    eval_batch_size=64,
    beam_size=20,
    top_k=10,
    d_kv=64,
    device="cuda"
)

## 5. Split data with leave-one-out

`loo_split()` performs a leave-one-out split on the interactions DataFrame:
- **train**: all interactions except the last 2 per user
- **val**: all interactions except the last 1 per user
- **test**: all interactions

Users with fewer than 3 interactions are dropped.

In [10]:
train_df, val_df, test_df = loo_split(interactions)

print(f"Train: {len(train_df):,} interactions, {train_df['user_id'].nunique():,} users")
print(f"Val:   {len(val_df):,} interactions, {val_df['user_id'].nunique():,} users")
print(f"Test:  {len(test_df):,} interactions, {test_df['user_id'].nunique():,} users")

Train: 9,704,103 interactions, 136,674 users
Val:   9,840,777 interactions, 136,674 users
Test:  9,977,451 interactions, 136,674 users


## 6. Train the model

`fit()` accepts pre-split train and validation DataFrames.
You control the split strategy -- the model does not split internally.

In [11]:
tiger.fit(train_df, val_df)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA H100 80GB HBM3 MIG 2g.20gb') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type     | Params | Mode 
-------------------------------------------
0 |

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=3` reached.


## 7. Evaluate the model

`evaluate()` computes ranking metrics (Hit@k, NDCG@k, MRR@k) on a test set.
For each user, the last item in the sequence is treated as the ground-truth target.

In [12]:
results = tiger.evaluate(test_df)

for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        Coverage@10        │   0.0023603341542184353   │
│          Gini@10          │    0.7614988684654236     │
│          Hit@10           │    0.0728229209780693     │
│          MRR@10           │   0.024664802476763725    │
│          NDCG@10          │    0.03576871380209923    │
└───────────────────────────┴───────────────────────────┘

Hit@10: 0.0728
NDCG@10: 0.0358
MRR@10: 0.0247
Gini@10: 0.7615
Coverage@10: 0.0024


## 8. Generate recommendations

`predict()` takes a DataFrame of user interaction histories and returns
a DataFrame of recommendations with columns: `user_id`, `item_id`, `score`, `rank`.

We can join with movie metadata to see the actual titles.

In [13]:
sample_users = interactions["user_id"].unique()[:5]
sample_interactions = interactions[interactions["user_id"].isin(sample_users)]

recommendations = tiger.predict(sample_interactions, top_k=5)
recommendations.merge(movies[["item_id", "title"]], on="item_id", how="left")

,user_id,item_id,score,rank,title
0,1,5952,-4.595025,1,"Lord of the Rings: The Two Towers, The (2002)"
1,1,3578,-4.673381,2,Gladiator (2000)
2,1,59315,-4.751242,3,Iron Man (2008)
3,1,6539,-4.858047,4,Pirates of the Caribbean: The Curse of the Bla...
4,1,5349,-5.812282,5,Spider-Man (2002)
5,2,1196,-3.895408,1,Star Wars: Episode V - The Empire Strikes Back...
6,2,260,-4.124645,2,Star Wars: Episode IV - A New Hope (1977)
7,2,1210,-4.179870,3,Star Wars: Episode VI - Return of the Jedi (1983)
8,2,541,-4.414659,4,Blade Runner (1982)
9,2,1240,-4.478841,5,"Terminator, The (1984)"


## 9. Save and load the model

`save()` writes the tokenizer, model weights, and config to a directory.
`load()` reconstructs the full `TIGERModel` from that directory.

In [14]:
import tempfile
import os

with tempfile.TemporaryDirectory() as tmpdir:
    save_path = os.path.join(tmpdir, "tiger_model")
    tiger.save(save_path)
    print(f"Saved to {save_path}")
    print(f"Contents: {os.listdir(save_path)}")

    loaded_tiger = TIGERModel.load(save_path)
    print(f"\nLoaded model with {len(loaded_tiger.tokenizer)} item SIDs")
    print(f"Codebook sizes: {loaded_tiger.codebook_sizes}")

Saved to /tmp/tmp2mudyxqd/tiger_model
Contents: ['tokenizer.pt', 'model.pt', 'config.json']


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL rectools.semantic.tokenizer.rqvae.RQVAE was not an allowed global by default. Please use `torch.serialization.add_safe_globals([rectools.semantic.tokenizer.rqvae.RQVAE])` or the `torch.serialization.safe_globals([rectools.semantic.tokenizer.rqvae.RQVAE])` context manager to allowlist this global if you trust this class/function.

Check the documen

## Summary

The TIGER pipeline in RecTools:

| Step | Function / Method | Input | Output |
|------|-------------------|-------|--------|
| Filtering | `k_core(df)` | Interactions DataFrame | Filtered DataFrame |
| Embedding | `encode_texts(texts, model)` | List of text strings | numpy embedding matrix |
| Tokenization | `SIDTokenizer(...).fit(ids, embs)` | Item IDs + embeddings | `SIDTokenizer` |
| Splitting | `loo_split(df)` | Interactions DataFrame | train, val, test DataFrames |
| Model creation | `TIGERModel(tokenizer)` | `SIDTokenizer` + hyperparams | `TIGERModel` |
| Training | `model.fit(train, val)` | Train + val DataFrames | Trained model (in-place) |
| Evaluation | `model.evaluate(test)` | Test DataFrame | Metrics dict |
| Prediction | `model.predict(df)` | Interactions DataFrame | Recommendations DataFrame |
| Persistence | `model.save(dir)` / `TIGERModel.load(dir)` | Directory path | Saved/loaded model |